# Task 5: Evaluate a Diffusion Model

This notebook evaluates the performance of a diffusion model by calculating the **PSNR (Peak Signal-to-Noise Ratio)** between original and denoised images.

## Overview

**Diffusion models** are a class of generative models that learn to denoise images by reversing a gradual noising process. They work by:
1. **Forward process**: Gradually adding Gaussian noise to images over multiple timesteps
2. **Reverse process**: Learning to remove noise step-by-step to recover the original image

**PSNR (Peak Signal-to-Noise Ratio)** is a metric used to measure the quality of reconstruction:
- Higher PSNR indicates better image quality (closer to original)
- PSNR is measured in decibels (dB)
- Typical values: 20-25 dB (acceptable), 25-30 dB (good), >30 dB (excellent)

## 1. Setup and Imports

In [ ]:
# Install required packages
!pip install torch torchvision numpy matplotlib scikit-image pillow tqdm --quiet

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
from PIL import Image
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 2. Define Diffusion Model Components

### 2.1 Noise Schedule

The noise schedule defines how noise is added over time. We use a linear schedule for beta values.

In [ ]:
class DiffusionSchedule:
    """Manages the noise schedule for diffusion process."""
    
    def __init__(self, num_timesteps=1000, beta_start=1e-4, beta_end=0.02):
        self.num_timesteps = num_timesteps
        
        # Linear beta schedule
        self.betas = torch.linspace(beta_start, beta_end, num_timesteps)
        
        # Pre-compute useful quantities
        self.alphas = 1.0 - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0)
        self.alphas_cumprod_prev = F.pad(self.alphas_cumprod[:-1], (1, 0), value=1.0)
        
        # Calculations for diffusion q(x_t | x_{t-1})
        self.sqrt_alphas_cumprod = torch.sqrt(self.alphas_cumprod)
        self.sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - self.alphas_cumprod)
        
        # Calculations for posterior q(x_{t-1} | x_t, x_0)
        self.posterior_variance = self.betas * (1.0 - self.alphas_cumprod_prev) / (1.0 - self.alphas_cumprod)
        self.sqrt_recip_alphas = torch.sqrt(1.0 / self.alphas)
        
    def to(self, device):
        """Move all tensors to specified device."""
        self.betas = self.betas.to(device)
        self.alphas = self.alphas.to(device)
        self.alphas_cumprod = self.alphas_cumprod.to(device)
        self.alphas_cumprod_prev = self.alphas_cumprod_prev.to(device)
        self.sqrt_alphas_cumprod = self.sqrt_alphas_cumprod.to(device)
        self.sqrt_one_minus_alphas_cumprod = self.sqrt_one_minus_alphas_cumprod.to(device)
        self.posterior_variance = self.posterior_variance.to(device)
        self.sqrt_recip_alphas = self.sqrt_recip_alphas.to(device)
        return self

# Create noise schedule
NUM_TIMESTEPS = 1000
schedule = DiffusionSchedule(num_timesteps=NUM_TIMESTEPS)
schedule = schedule.to(device)
print(f"Diffusion schedule created with {NUM_TIMESTEPS} timesteps")

### 2.2 Forward Diffusion Process

The forward process adds noise to the image according to the schedule.

In [ ]:
def forward_diffusion(x_0, t, schedule, noise=None):
    """
    Apply forward diffusion to add noise to image.
    
    Args:
        x_0: Original image tensor [B, C, H, W]
        t: Timestep tensor [B]
        schedule: DiffusionSchedule instance
        noise: Optional pre-generated noise
    
    Returns:
        x_t: Noisy image at timestep t
        noise: The noise that was added
    """
    if noise is None:
        noise = torch.randn_like(x_0)
    
    # Get schedule values for timestep t
    sqrt_alpha_cumprod = schedule.sqrt_alphas_cumprod[t].view(-1, 1, 1, 1)
    sqrt_one_minus_alpha_cumprod = schedule.sqrt_one_minus_alphas_cumprod[t].view(-1, 1, 1, 1)
    
    # Apply forward diffusion: x_t = sqrt(alpha_bar_t) * x_0 + sqrt(1 - alpha_bar_t) * noise
    x_t = sqrt_alpha_cumprod * x_0 + sqrt_one_minus_alpha_cumprod * noise
    
    return x_t, noise

print("Forward diffusion function defined")

### 2.3 U-Net Denoising Model

A simplified U-Net architecture for noise prediction.

In [ ]:
class SinusoidalPositionEmbedding(nn.Module):
    """Sinusoidal embeddings for timestep conditioning."""
    
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
        
    def forward(self, t):
        device = t.device
        half_dim = self.dim // 2
        embeddings = np.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=device) * -embeddings)
        embeddings = t[:, None] * embeddings[None, :]
        embeddings = torch.cat([torch.sin(embeddings), torch.cos(embeddings)], dim=-1)
        return embeddings


class ConvBlock(nn.Module):
    """Convolutional block with GroupNorm and SiLU activation."""
    
    def __init__(self, in_ch, out_ch, time_emb_dim=None):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.norm1 = nn.GroupNorm(8, out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.norm2 = nn.GroupNorm(8, out_ch)
        self.act = nn.SiLU()
        
        if time_emb_dim is not None:
            self.time_mlp = nn.Linear(time_emb_dim, out_ch)
        else:
            self.time_mlp = None
            
        if in_ch != out_ch:
            self.shortcut = nn.Conv2d(in_ch, out_ch, 1)
        else:
            self.shortcut = nn.Identity()
            
    def forward(self, x, t_emb=None):
        h = self.conv1(x)
        h = self.norm1(h)
        h = self.act(h)
        
        if self.time_mlp is not None and t_emb is not None:
            h = h + self.time_mlp(self.act(t_emb))[:, :, None, None]
            
        h = self.conv2(h)
        h = self.norm2(h)
        h = self.act(h)
        
        return h + self.shortcut(x)


class SimpleUNet(nn.Module):
    """Simplified U-Net for diffusion model denoising."""
    
    def __init__(self, in_channels=3, out_channels=3, base_channels=64, time_emb_dim=256):
        super().__init__()
        
        # Time embedding
        self.time_embed = nn.Sequential(
            SinusoidalPositionEmbedding(time_emb_dim),
            nn.Linear(time_emb_dim, time_emb_dim),
            nn.SiLU(),
            nn.Linear(time_emb_dim, time_emb_dim)
        )
        
        # Encoder
        self.enc1 = ConvBlock(in_channels, base_channels, time_emb_dim)
        self.enc2 = ConvBlock(base_channels, base_channels * 2, time_emb_dim)
        self.enc3 = ConvBlock(base_channels * 2, base_channels * 4, time_emb_dim)
        
        self.pool = nn.MaxPool2d(2)
        
        # Bottleneck
        self.bottleneck = ConvBlock(base_channels * 4, base_channels * 8, time_emb_dim)
        
        # Decoder
        self.up3 = nn.ConvTranspose2d(base_channels * 8, base_channels * 4, 2, stride=2)
        self.dec3 = ConvBlock(base_channels * 8, base_channels * 4, time_emb_dim)
        
        self.up2 = nn.ConvTranspose2d(base_channels * 4, base_channels * 2, 2, stride=2)
        self.dec2 = ConvBlock(base_channels * 4, base_channels * 2, time_emb_dim)
        
        self.up1 = nn.ConvTranspose2d(base_channels * 2, base_channels, 2, stride=2)
        self.dec1 = ConvBlock(base_channels * 2, base_channels, time_emb_dim)
        
        # Output
        self.out = nn.Conv2d(base_channels, out_channels, 1)
        
    def forward(self, x, t):
        # Time embedding
        t_emb = self.time_embed(t)
        
        # Encoder
        e1 = self.enc1(x, t_emb)
        e2 = self.enc2(self.pool(e1), t_emb)
        e3 = self.enc3(self.pool(e2), t_emb)
        
        # Bottleneck
        b = self.bottleneck(self.pool(e3), t_emb)
        
        # Decoder with skip connections
        d3 = self.dec3(torch.cat([self.up3(b), e3], dim=1), t_emb)
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1), t_emb)
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1), t_emb)
        
        return self.out(d1)

# Create model
model = SimpleUNet(in_channels=3, out_channels=3, base_channels=64).to(device)
print(f"U-Net model created with {sum(p.numel() for p in model.parameters()):,} parameters")

### 2.4 Reverse Diffusion (Denoising) Process

In [ ]:
@torch.no_grad()
def reverse_diffusion_step(model, x_t, t, schedule):
    """
    Perform one step of reverse diffusion (denoising).
    
    Args:
        model: Noise prediction model
        x_t: Noisy image at timestep t
        t: Current timestep
        schedule: DiffusionSchedule instance
    
    Returns:
        x_{t-1}: Denoised image at timestep t-1
    """
    batch_size = x_t.shape[0]
    t_tensor = torch.full((batch_size,), t, device=x_t.device, dtype=torch.long)
    
    # Predict noise
    predicted_noise = model(x_t, t_tensor)
    
    # Get schedule values
    beta_t = schedule.betas[t]
    sqrt_one_minus_alpha_cumprod_t = schedule.sqrt_one_minus_alphas_cumprod[t]
    sqrt_recip_alpha_t = schedule.sqrt_recip_alphas[t]
    
    # Compute mean of p(x_{t-1} | x_t)
    model_mean = sqrt_recip_alpha_t * (x_t - beta_t / sqrt_one_minus_alpha_cumprod_t * predicted_noise)
    
    if t == 0:
        return model_mean
    else:
        # Add noise for timesteps > 0
        posterior_variance_t = schedule.posterior_variance[t]
        noise = torch.randn_like(x_t)
        return model_mean + torch.sqrt(posterior_variance_t) * noise


@torch.no_grad()
def denoise_image(model, x_noisy, start_timestep, schedule, num_steps=None):
    """
    Denoise an image from a given timestep back to clean image.
    
    Args:
        model: Noise prediction model
        x_noisy: Noisy image
        start_timestep: Starting timestep for denoising
        schedule: DiffusionSchedule instance
        num_steps: Number of denoising steps (if None, use all steps from start_timestep)
    
    Returns:
        Denoised image
    """
    x = x_noisy.clone()
    
    if num_steps is None:
        timesteps = range(start_timestep, -1, -1)
    else:
        # Use fewer steps with striding
        step_size = max(1, start_timestep // num_steps)
        timesteps = range(start_timestep, -1, -step_size)
    
    for t in timesteps:
        x = reverse_diffusion_step(model, x, t, schedule)
    
    return x

print("Reverse diffusion functions defined")

## 3. PSNR Calculation Functions

PSNR is calculated as:

$$PSNR = 10 \cdot \log_{10}\left(\frac{MAX^2}{MSE}\right)$$

Where MAX is the maximum possible pixel value and MSE is the Mean Squared Error.

In [ ]:
def calculate_psnr(original, reconstructed, data_range=1.0):
    """
    Calculate PSNR between original and reconstructed images.
    
    Args:
        original: Original image (numpy array or torch tensor)
        reconstructed: Reconstructed image (numpy array or torch tensor)
        data_range: Range of the data (1.0 for [0,1] normalized, 255 for [0,255])
    
    Returns:
        PSNR value in dB
    """
    # Convert to numpy if tensor
    if torch.is_tensor(original):
        original = original.cpu().numpy()
    if torch.is_tensor(reconstructed):
        reconstructed = reconstructed.cpu().numpy()
    
    # Clip values to valid range
    original = np.clip(original, 0, data_range)
    reconstructed = np.clip(reconstructed, 0, data_range)
    
    return psnr(original, reconstructed, data_range=data_range)


def calculate_psnr_batch(original_batch, reconstructed_batch, data_range=1.0):
    """
    Calculate PSNR for a batch of images.
    
    Args:
        original_batch: Batch of original images [B, C, H, W]
        reconstructed_batch: Batch of reconstructed images [B, C, H, W]
        data_range: Range of the data
    
    Returns:
        List of PSNR values and mean PSNR
    """
    psnr_values = []
    
    for orig, recon in zip(original_batch, reconstructed_batch):
        # Convert from [C, H, W] to [H, W, C] for skimage
        if torch.is_tensor(orig):
            orig = orig.permute(1, 2, 0).cpu().numpy()
            recon = recon.permute(1, 2, 0).cpu().numpy()
        
        orig = np.clip(orig, 0, data_range)
        recon = np.clip(recon, 0, data_range)
        
        psnr_val = psnr(orig, recon, data_range=data_range)
        psnr_values.append(psnr_val)
    
    return psnr_values, np.mean(psnr_values)


def calculate_ssim_batch(original_batch, reconstructed_batch, data_range=1.0):
    """
    Calculate SSIM (Structural Similarity) for a batch of images.
    
    Args:
        original_batch: Batch of original images [B, C, H, W]
        reconstructed_batch: Batch of reconstructed images [B, C, H, W]
        data_range: Range of the data
    
    Returns:
        List of SSIM values and mean SSIM
    """
    ssim_values = []
    
    for orig, recon in zip(original_batch, reconstructed_batch):
        # Convert from [C, H, W] to [H, W, C] for skimage
        if torch.is_tensor(orig):
            orig = orig.permute(1, 2, 0).cpu().numpy()
            recon = recon.permute(1, 2, 0).cpu().numpy()
        
        orig = np.clip(orig, 0, data_range)
        recon = np.clip(recon, 0, data_range)
        
        # Determine win_size based on image dimensions
        min_dim = min(orig.shape[0], orig.shape[1])
        win_size = min(7, min_dim if min_dim % 2 == 1 else min_dim - 1)
        
        ssim_val = ssim(orig, recon, data_range=data_range, channel_axis=2, win_size=win_size)
        ssim_values.append(ssim_val)
    
    return ssim_values, np.mean(ssim_values)

print("PSNR and SSIM calculation functions defined")

## 4. Load Dataset and Prepare Images

In [ ]:
# Image size for the model (must be divisible by 8 for U-Net)
IMG_SIZE = 32

# Define transforms
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),  # Converts to [0, 1] range
])

# Load CIFAR-10 dataset (small images for quick demonstration)
print("Loading CIFAR-10 dataset...")
test_dataset = datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

# Create data loader
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=True)

# Get a sample batch
sample_images, sample_labels = next(iter(test_loader))
sample_images = sample_images.to(device)

print(f"Dataset loaded: {len(test_dataset)} test images")
print(f"Sample batch shape: {sample_images.shape}")
print(f"Value range: [{sample_images.min():.3f}, {sample_images.max():.3f}]")

In [ ]:
# Visualize sample images
def show_images(images, title="Images", nrow=4):
    """Display a grid of images."""
    n_images = min(len(images), nrow * 2)
    fig, axes = plt.subplots(2, nrow, figsize=(nrow * 2.5, 5))
    
    for i, ax in enumerate(axes.flat):
        if i < n_images:
            img = images[i]
            if torch.is_tensor(img):
                img = img.cpu().permute(1, 2, 0).numpy()
            img = np.clip(img, 0, 1)
            ax.imshow(img)
        ax.axis('off')
    
    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()

show_images(sample_images, "Original CIFAR-10 Images")

## 5. Train the Diffusion Model

Training the U-Net to predict noise added to images.

In [ ]:
def train_diffusion_model(model, dataloader, schedule, num_epochs=5, lr=1e-4):
    """
    Train the diffusion model.
    
    Args:
        model: U-Net model
        dataloader: Training data loader
        schedule: DiffusionSchedule instance
        num_epochs: Number of training epochs
        lr: Learning rate
    
    Returns:
        Training losses
    """
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    
    losses = []
    
    model.train()
    for epoch in range(num_epochs):
        epoch_losses = []
        
        pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs}")
        for batch_idx, (images, _) in enumerate(pbar):
            images = images.to(device)
            batch_size = images.shape[0]
            
            # Sample random timesteps
            t = torch.randint(0, schedule.num_timesteps, (batch_size,), device=device)
            
            # Add noise
            noise = torch.randn_like(images)
            noisy_images, _ = forward_diffusion(images, t, schedule, noise)
            
            # Predict noise
            predicted_noise = model(noisy_images, t)
            
            # Calculate loss
            loss = criterion(predicted_noise, noise)
            
            # Backpropagation
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            epoch_losses.append(loss.item())
            pbar.set_postfix({'loss': f"{loss.item():.4f}"})
            
            # Limit batches for demonstration
            if batch_idx >= 100:
                break
        
        avg_loss = np.mean(epoch_losses)
        losses.append(avg_loss)
        print(f"Epoch {epoch+1}/{num_epochs} - Average Loss: {avg_loss:.4f}")
    
    return losses

# Train the model (reduced epochs for demonstration)
print("\nTraining diffusion model...")
print("(This is a simplified training for demonstration purposes)\n")

train_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)
training_losses = train_diffusion_model(model, train_loader, schedule, num_epochs=3, lr=1e-4)

In [ ]:
# Plot training loss
plt.figure(figsize=(10, 4))
plt.plot(training_losses, 'b-', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Diffusion Model Training Loss')
plt.grid(True, alpha=0.3)
plt.show()

## 6. Evaluate Diffusion Model with PSNR

Now we evaluate the model's denoising capability by:
1. Adding noise to original images at various timesteps
2. Denoising using the trained model
3. Calculating PSNR between original and denoised images

In [ ]:
@torch.no_grad()
def evaluate_diffusion_psnr(model, images, schedule, timesteps_to_test=[50, 100, 200, 500]):
    """
    Evaluate diffusion model denoising performance using PSNR.
    
    Args:
        model: Trained diffusion model
        images: Batch of original images
        schedule: DiffusionSchedule instance
        timesteps_to_test: List of noise timesteps to test
    
    Returns:
        Dictionary with evaluation results
    """
    model.eval()
    results = {
        'timesteps': [],
        'psnr_noisy': [],
        'psnr_denoised': [],
        'ssim_noisy': [],
        'ssim_denoised': [],
        'improvement': []
    }
    
    print("Evaluating diffusion model at different noise levels...\n")
    print(f"{'Timestep':>10} {'Noisy PSNR':>12} {'Denoised PSNR':>14} {'Improvement':>12} {'SSIM Improvement':>16}")
    print("-" * 70)
    
    for t in timesteps_to_test:
        # Create timestep tensor
        t_tensor = torch.full((images.shape[0],), t, device=device, dtype=torch.long)
        
        # Add noise
        noisy_images, noise = forward_diffusion(images, t_tensor, schedule)
        
        # Denoise
        denoised_images = denoise_image(model, noisy_images, t, schedule, num_steps=min(t, 50))
        
        # Calculate PSNR
        _, psnr_noisy = calculate_psnr_batch(images, noisy_images)
        _, psnr_denoised = calculate_psnr_batch(images, denoised_images)
        
        # Calculate SSIM
        _, ssim_noisy = calculate_ssim_batch(images, noisy_images)
        _, ssim_denoised = calculate_ssim_batch(images, denoised_images)
        
        improvement = psnr_denoised - psnr_noisy
        ssim_improvement = ssim_denoised - ssim_noisy
        
        results['timesteps'].append(t)
        results['psnr_noisy'].append(psnr_noisy)
        results['psnr_denoised'].append(psnr_denoised)
        results['ssim_noisy'].append(ssim_noisy)
        results['ssim_denoised'].append(ssim_denoised)
        results['improvement'].append(improvement)
        
        print(f"{t:>10} {psnr_noisy:>12.2f} dB {psnr_denoised:>14.2f} dB {improvement:>+12.2f} dB {ssim_improvement:>+16.4f}")
    
    return results

# Evaluate at different timesteps
evaluation_results = evaluate_diffusion_psnr(
    model, 
    sample_images, 
    schedule, 
    timesteps_to_test=[25, 50, 100, 200, 400, 600]
)

In [ ]:
# Visualize PSNR results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PSNR comparison
ax1 = axes[0]
x = range(len(evaluation_results['timesteps']))
width = 0.35

bars1 = ax1.bar([i - width/2 for i in x], evaluation_results['psnr_noisy'], width, label='Noisy', color='red', alpha=0.7)
bars2 = ax1.bar([i + width/2 for i in x], evaluation_results['psnr_denoised'], width, label='Denoised', color='green', alpha=0.7)

ax1.set_xlabel('Noise Timestep')
ax1.set_ylabel('PSNR (dB)')
ax1.set_title('PSNR: Noisy vs Denoised Images')
ax1.set_xticks(x)
ax1.set_xticklabels(evaluation_results['timesteps'])
ax1.legend()
ax1.grid(True, alpha=0.3)

# Add quality reference lines
ax1.axhline(y=30, color='blue', linestyle='--', alpha=0.5, label='Excellent (30 dB)')
ax1.axhline(y=25, color='orange', linestyle='--', alpha=0.5, label='Good (25 dB)')
ax1.axhline(y=20, color='purple', linestyle='--', alpha=0.5, label='Acceptable (20 dB)')

# PSNR improvement
ax2 = axes[1]
colors = ['green' if imp > 0 else 'red' for imp in evaluation_results['improvement']]
ax2.bar(evaluation_results['timesteps'], evaluation_results['improvement'], color=colors, alpha=0.7)
ax2.set_xlabel('Noise Timestep')
ax2.set_ylabel('PSNR Improvement (dB)')
ax2.set_title('PSNR Improvement After Denoising')
ax2.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Visual Comparison: Original vs Noisy vs Denoised

In [ ]:
@torch.no_grad()
def visualize_denoising(model, images, schedule, timestep=100, num_images=4):
    """
    Visualize the denoising process with PSNR values.
    
    Args:
        model: Trained diffusion model
        images: Batch of original images
        schedule: DiffusionSchedule instance
        timestep: Noise timestep to use
        num_images: Number of images to display
    """
    model.eval()
    
    # Select subset of images
    images = images[:num_images]
    
    # Create timestep tensor
    t_tensor = torch.full((images.shape[0],), timestep, device=device, dtype=torch.long)
    
    # Add noise
    noisy_images, _ = forward_diffusion(images, t_tensor, schedule)
    
    # Denoise
    denoised_images = denoise_image(model, noisy_images, timestep, schedule, num_steps=min(timestep, 50))
    
    # Calculate PSNR for each image
    psnr_noisy, _ = calculate_psnr_batch(images, noisy_images)
    psnr_denoised, _ = calculate_psnr_batch(images, denoised_images)
    
    # Create visualization
    fig, axes = plt.subplots(3, num_images, figsize=(num_images * 3, 9))
    
    for i in range(num_images):
        # Original
        orig = images[i].cpu().permute(1, 2, 0).numpy()
        axes[0, i].imshow(np.clip(orig, 0, 1))
        axes[0, i].set_title(f'Original', fontsize=10)
        axes[0, i].axis('off')
        
        # Noisy
        noisy = noisy_images[i].cpu().permute(1, 2, 0).numpy()
        axes[1, i].imshow(np.clip(noisy, 0, 1))
        axes[1, i].set_title(f'Noisy (t={timestep})\nPSNR: {psnr_noisy[i]:.2f} dB', fontsize=10)
        axes[1, i].axis('off')
        
        # Denoised
        denoised = denoised_images[i].cpu().permute(1, 2, 0).numpy()
        axes[2, i].imshow(np.clip(denoised, 0, 1))
        axes[2, i].set_title(f'Denoised\nPSNR: {psnr_denoised[i]:.2f} dB', fontsize=10)
        axes[2, i].axis('off')
    
    # Add row labels
    axes[0, 0].set_ylabel('Original', fontsize=12, rotation=0, labelpad=60)
    axes[1, 0].set_ylabel('Noisy', fontsize=12, rotation=0, labelpad=60)
    axes[2, 0].set_ylabel('Denoised', fontsize=12, rotation=0, labelpad=60)
    
    plt.suptitle(f'Diffusion Model Denoising (Timestep={timestep})', fontsize=14)
    plt.tight_layout()
    plt.show()

# Visualize denoising at different timesteps
for t in [50, 200, 500]:
    print(f"\n{'='*50}")
    print(f"Visualization at timestep t={t}")
    print(f"{'='*50}")
    visualize_denoising(model, sample_images, schedule, timestep=t, num_images=4)

## 8. Detailed PSNR Analysis

In [ ]:
@torch.no_grad()
def detailed_psnr_analysis(model, dataloader, schedule, timestep=100, num_batches=5):
    """
    Perform detailed PSNR analysis across multiple batches.
    
    Args:
        model: Trained diffusion model
        dataloader: Data loader for evaluation
        schedule: DiffusionSchedule instance
        timestep: Noise timestep to evaluate
        num_batches: Number of batches to evaluate
    
    Returns:
        Dictionary with detailed statistics
    """
    model.eval()
    
    all_psnr_noisy = []
    all_psnr_denoised = []
    all_ssim_noisy = []
    all_ssim_denoised = []
    
    print(f"Analyzing PSNR at timestep t={timestep}...")
    
    for batch_idx, (images, _) in enumerate(tqdm(dataloader, total=num_batches)):
        if batch_idx >= num_batches:
            break
            
        images = images.to(device)
        t_tensor = torch.full((images.shape[0],), timestep, device=device, dtype=torch.long)
        
        # Add noise and denoise
        noisy_images, _ = forward_diffusion(images, t_tensor, schedule)
        denoised_images = denoise_image(model, noisy_images, timestep, schedule, num_steps=min(timestep, 50))
        
        # Calculate metrics
        psnr_n, _ = calculate_psnr_batch(images, noisy_images)
        psnr_d, _ = calculate_psnr_batch(images, denoised_images)
        ssim_n, _ = calculate_ssim_batch(images, noisy_images)
        ssim_d, _ = calculate_ssim_batch(images, denoised_images)
        
        all_psnr_noisy.extend(psnr_n)
        all_psnr_denoised.extend(psnr_d)
        all_ssim_noisy.extend(ssim_n)
        all_ssim_denoised.extend(ssim_d)
    
    # Calculate statistics
    stats = {
        'timestep': timestep,
        'num_samples': len(all_psnr_noisy),
        'psnr_noisy': {
            'mean': np.mean(all_psnr_noisy),
            'std': np.std(all_psnr_noisy),
            'min': np.min(all_psnr_noisy),
            'max': np.max(all_psnr_noisy)
        },
        'psnr_denoised': {
            'mean': np.mean(all_psnr_denoised),
            'std': np.std(all_psnr_denoised),
            'min': np.min(all_psnr_denoised),
            'max': np.max(all_psnr_denoised)
        },
        'ssim_noisy': {
            'mean': np.mean(all_ssim_noisy),
            'std': np.std(all_ssim_noisy)
        },
        'ssim_denoised': {
            'mean': np.mean(all_ssim_denoised),
            'std': np.std(all_ssim_denoised)
        },
        'improvement': np.mean(all_psnr_denoised) - np.mean(all_psnr_noisy),
        'all_psnr_noisy': all_psnr_noisy,
        'all_psnr_denoised': all_psnr_denoised
    }
    
    return stats

# Perform detailed analysis
stats = detailed_psnr_analysis(model, test_loader, schedule, timestep=100, num_batches=10)

print(f"\n{'='*60}")
print(f"DETAILED PSNR ANALYSIS RESULTS (Timestep={stats['timestep']})")
print(f"{'='*60}")
print(f"Number of samples analyzed: {stats['num_samples']}")
print(f"\nNoisy Images:")
print(f"  PSNR: {stats['psnr_noisy']['mean']:.2f} ± {stats['psnr_noisy']['std']:.2f} dB")
print(f"  Range: [{stats['psnr_noisy']['min']:.2f}, {stats['psnr_noisy']['max']:.2f}] dB")
print(f"  SSIM: {stats['ssim_noisy']['mean']:.4f} ± {stats['ssim_noisy']['std']:.4f}")
print(f"\nDenoised Images:")
print(f"  PSNR: {stats['psnr_denoised']['mean']:.2f} ± {stats['psnr_denoised']['std']:.2f} dB")
print(f"  Range: [{stats['psnr_denoised']['min']:.2f}, {stats['psnr_denoised']['max']:.2f}] dB")
print(f"  SSIM: {stats['ssim_denoised']['mean']:.4f} ± {stats['ssim_denoised']['std']:.4f}")
print(f"\nPSNR Improvement: {stats['improvement']:+.2f} dB")

In [ ]:
# Visualize PSNR distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of PSNR values
ax1 = axes[0]
ax1.hist(stats['all_psnr_noisy'], bins=20, alpha=0.7, label='Noisy', color='red')
ax1.hist(stats['all_psnr_denoised'], bins=20, alpha=0.7, label='Denoised', color='green')
ax1.axvline(stats['psnr_noisy']['mean'], color='darkred', linestyle='--', linewidth=2, label=f'Noisy Mean: {stats["psnr_noisy"]["mean"]:.2f} dB')
ax1.axvline(stats['psnr_denoised']['mean'], color='darkgreen', linestyle='--', linewidth=2, label=f'Denoised Mean: {stats["psnr_denoised"]["mean"]:.2f} dB')
ax1.set_xlabel('PSNR (dB)')
ax1.set_ylabel('Frequency')
ax1.set_title('PSNR Distribution: Noisy vs Denoised')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Scatter plot of noisy vs denoised PSNR
ax2 = axes[1]
ax2.scatter(stats['all_psnr_noisy'], stats['all_psnr_denoised'], alpha=0.5, s=20)
min_val = min(min(stats['all_psnr_noisy']), min(stats['all_psnr_denoised']))
max_val = max(max(stats['all_psnr_noisy']), max(stats['all_psnr_denoised']))
ax2.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='No improvement line')
ax2.set_xlabel('Noisy PSNR (dB)')
ax2.set_ylabel('Denoised PSNR (dB)')
ax2.set_title('Noisy vs Denoised PSNR')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Summary and Conclusions

In [ ]:
print("="*70)
print("DIFFUSION MODEL EVALUATION SUMMARY")
print("="*70)

print("\n1. MODEL ARCHITECTURE:")
print(f"   - Type: U-Net with time embeddings")
print(f"   - Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"   - Input/Output: {IMG_SIZE}x{IMG_SIZE} RGB images")

print("\n2. DIFFUSION PROCESS:")
print(f"   - Number of timesteps: {NUM_TIMESTEPS}")
print(f"   - Noise schedule: Linear beta from 1e-4 to 0.02")

print("\n3. EVALUATION METRICS:")
print(f"   - Primary: PSNR (Peak Signal-to-Noise Ratio)")
print(f"   - Secondary: SSIM (Structural Similarity Index)")

print("\n4. KEY FINDINGS:")
print(f"   - At timestep t=100:")
print(f"     * Noisy PSNR: {stats['psnr_noisy']['mean']:.2f} dB")
print(f"     * Denoised PSNR: {stats['psnr_denoised']['mean']:.2f} dB")
print(f"     * PSNR Improvement: {stats['improvement']:+.2f} dB")

print("\n5. PSNR QUALITY REFERENCE:")
print("   - > 30 dB: Excellent quality")
print("   - 25-30 dB: Good quality")
print("   - 20-25 dB: Acceptable quality")
print("   - < 20 dB: Poor quality")

print("\n6. OBSERVATIONS:")
if stats['improvement'] > 0:
    print(f"   ✓ Model successfully improves image quality after denoising")
    print(f"   ✓ Average PSNR improvement: {stats['improvement']:.2f} dB")
else:
    print(f"   ✗ Model needs more training to effectively denoise images")
print(f"   - Higher timesteps = more noise = lower initial PSNR")
print(f"   - Model learns to reverse the diffusion process")

print("\n" + "="*70)
print("Evaluation complete!")
print("="*70)

## 10. Additional Experiments

### Evaluate with custom images (if needed)

In [ ]:
def evaluate_custom_image(model, image_path_or_tensor, schedule, timestep=100):
    """
    Evaluate the diffusion model on a custom image.
    
    Args:
        model: Trained diffusion model
        image_path_or_tensor: Path to image or tensor
        schedule: DiffusionSchedule instance
        timestep: Noise timestep
    
    Returns:
        Dictionary with PSNR results
    """
    model.eval()
    
    # Load image if path provided
    if isinstance(image_path_or_tensor, str):
        img = Image.open(image_path_or_tensor).convert('RGB')
        transform = transforms.Compose([
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.ToTensor()
        ])
        image = transform(img).unsqueeze(0).to(device)
    else:
        image = image_path_or_tensor.to(device)
        if image.dim() == 3:
            image = image.unsqueeze(0)
    
    with torch.no_grad():
        # Add noise
        t_tensor = torch.full((1,), timestep, device=device, dtype=torch.long)
        noisy_image, _ = forward_diffusion(image, t_tensor, schedule)
        
        # Denoise
        denoised_image = denoise_image(model, noisy_image, timestep, schedule, num_steps=min(timestep, 50))
        
        # Calculate PSNR
        psnr_noisy = calculate_psnr(
            image[0].permute(1, 2, 0).cpu().numpy(),
            noisy_image[0].permute(1, 2, 0).cpu().numpy()
        )
        psnr_denoised = calculate_psnr(
            image[0].permute(1, 2, 0).cpu().numpy(),
            denoised_image[0].permute(1, 2, 0).cpu().numpy()
        )
    
    # Visualize
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    
    axes[0].imshow(np.clip(image[0].cpu().permute(1, 2, 0).numpy(), 0, 1))
    axes[0].set_title('Original')
    axes[0].axis('off')
    
    axes[1].imshow(np.clip(noisy_image[0].cpu().permute(1, 2, 0).numpy(), 0, 1))
    axes[1].set_title(f'Noisy (t={timestep})\nPSNR: {psnr_noisy:.2f} dB')
    axes[1].axis('off')
    
    axes[2].imshow(np.clip(denoised_image[0].cpu().permute(1, 2, 0).numpy(), 0, 1))
    axes[2].set_title(f'Denoised\nPSNR: {psnr_denoised:.2f} dB')
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    return {
        'psnr_noisy': psnr_noisy,
        'psnr_denoised': psnr_denoised,
        'improvement': psnr_denoised - psnr_noisy
    }

# Example: Evaluate on a sample image from the test set
print("Example evaluation on a single image:")
result = evaluate_custom_image(model, sample_images[0], schedule, timestep=150)
print(f"\nResults:")
print(f"  Noisy PSNR: {result['psnr_noisy']:.2f} dB")
print(f"  Denoised PSNR: {result['psnr_denoised']:.2f} dB")
print(f"  Improvement: {result['improvement']:+.2f} dB")